In [2]:
pip install pymupdf

   ---------------------------------------- 0.0/18.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.7 MB ? eta -:--:--
    --------------------------------------- 0.3/18.7 MB ? eta -:--:--
   - -------------------------------------- 0.5/18.7 MB 1.2 MB/s eta 0:00:16
   - -------------------------------------- 0.8/18.7 MB 1.2 MB/s eta 0:00:15
   -- ------------------------------------- 1.0/18.7 MB 1.2 MB/s eta 0:00:15
   -- ------------------------------------- 1.3/18.7 MB 1.1 MB/s eta 0:00:17
   --- ------------------------------------ 1.6/18.7 MB 1.2 MB/s eta 0:00:15
   --- ------------------------------------ 1.6/18.7 MB 1.2 MB/s eta 0:00:15
   --- ------------------------------------ 1.6/18.7 MB 1.2 MB/s eta 0:00:15
   --- ------------------------------------ 1.6/18.7 MB 1.2 MB/s eta 0:00:15
   --- ------------------------------------ 1.8/18.7 MB 792.8 kB/s eta 0:00:22
   ---- -----------------


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: C:\Users\Dilun\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [9]:
import pathlib
import fitz  # PyMuPDF
from textwrap import wrap
import pandas as pd
from PIL import Image
import io
import numpy as np

pdf_path = pathlib.Path("../data/lectures/Cloud Computing - Lectures All in One.pdf")
image_dir = pathlib.Path("../data/lectures/images")
image_dir.mkdir(parents=True, exist_ok=True)

pages = []
with fitz.open(pdf_path) as doc:
    for page_index, page in enumerate(doc):
        for img_index, img in enumerate(page.get_images(full=True)):
            xref = img[0]
            base = doc.extract_image(xref)
            image_bytes = base["image"]
            with Image.open(io.BytesIO(image_bytes)) as im:
                
                gray = np.array(im.convert("L"))
                if gray.shape[0] * gray.shape[1] < 10_000:
                    continue  # skip tiny images
                if gray.std() < 8:
                    continue  # skip low-contrast images
                white_ratio = (gray > 240).mean()
                if white_ratio > 0.92:
                    continue  # skip almost all-white images
            ext = base["ext"]
            out_path = image_dir / f"page{page_index+1}_img{img_index+1}.{ext}"
            out_path.write_bytes(image_bytes)
        pages.append(page.get_text("text"))

text = "\n\n".join(pages)
clean_text = text.replace("\u00a0", " ").strip()
chunks = wrap(clean_text, 1200)

df = pd.DataFrame({"chunk_id": range(len(chunks)), "text": chunks})
df.head()
summary = "\n".join(f"Chunk {i}: {chunk[:160]}..." for i, chunk in enumerate(chunks[:5]))
print(summary)

Chunk 0: Cloud Computing Module Overview   Module Overview   Module Objectives Provide an in depth  understanding of  Cloud Computing  theory. 1 Provide hands-on- experi...
Chunk 1: Azure Source: https://viniciusdeschamps.com.br/azure-certification-journey/   Introduction to Cloud  Computing IT4090 – Cloud Computing   Lecture Outline • Evol...
Chunk 2: generators installed and distributed electricity to  customers over wires. • Those who needs electricity paid a monthly cost and used it. • Advantages • No need...
Chunk 3: for what  you provision as you  use) 2 Add / Remove  capacity as and when  required. 3 Destroy and stop  paying if you don’t  need it. 4   Cloud Computing and t...
Chunk 4: ubiquitous, convenient,  on-demand network access to a shared pool of configurable  computing resources (e.g., networks, servers, storage, applications,  and se...
